# 🤖 Machine Learning Basics
### Module Deliverable — Data Science Internship (Codomax)
**Author:** Mohd Anas  
**Date:** September 2026

---

This notebook covers the fundamentals of Machine Learning using Python and Scikit-Learn:

1. **Supervised Learning — Regression** (Linear Regression)
2. **Supervised Learning — Classification** (Logistic Regression, Decision Tree, Random Forest)
3. **Data Preprocessing** (Train/Test Split, Feature Scaling)
4. **Model Evaluation** (MSE, R², Accuracy, Confusion Matrix, Classification Report)
5. **Visualization** of results and model performance

---
## 1. Import Libraries

In [ ]:
# Core libraries
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-Learn — Datasets
from sklearn.datasets import load_iris, fetch_california_housing

# Scikit-Learn — Preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Scikit-Learn — Models
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Scikit-Learn — Evaluation
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, confusion_matrix, classification_report
)

# Settings
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print('✅ All libraries imported successfully!')

---
## 2. Part A — Supervised Learning: Regression
### Predicting California Housing Prices with Linear Regression

We'll use the **California Housing** dataset to predict median house values based on features like median income, house age, average rooms, etc.

### 2.1 Load & Explore the Dataset

In [ ]:
# Load the California Housing dataset
housing = fetch_california_housing()
df_housing = pd.DataFrame(housing.data, columns=housing.feature_names)
df_housing['MedHouseVal'] = housing.target  # Target variable (in $100,000s)

print(f'Dataset Shape: {df_housing.shape}')
print(f'Features: {list(housing.feature_names)}')
print(f'Target: Median House Value (in $100,000s)')
print()
df_housing.head(10)

In [ ]:
# Statistical summary
df_housing.describe().round(2)

In [ ]:
# Check for missing values
print('Missing values per column:')
print(df_housing.isnull().sum())
print(f'\nTotal missing: {df_housing.isnull().sum().sum()}')

### 2.2 Exploratory Data Analysis (EDA)

In [ ]:
# Distribution of the target variable
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df_housing['MedHouseVal'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Median House Value ($100k)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Median House Value')
axes[0].axvline(df_housing['MedHouseVal'].mean(), color='red', linestyle='--', label=f"Mean: ${df_housing['MedHouseVal'].mean()*100:.0f}k")
axes[0].legend()

# Median Income vs House Value scatter plot
axes[1].scatter(df_housing['MedInc'], df_housing['MedHouseVal'], alpha=0.1, s=5, color='steelblue')
axes[1].set_xlabel('Median Income')
axes[1].set_ylabel('Median House Value ($100k)')
axes[1].set_title('Median Income vs. House Value')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(10, 8))
correlation = df_housing.corr().round(2)
mask = np.triu(np.ones_like(correlation, dtype=bool))
sns.heatmap(correlation, annot=True, cmap='RdBu_r', center=0, mask=mask,
            square=True, linewidths=0.5, fmt='.2f')
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('\n📊 Key Insight: MedInc (Median Income) has the strongest positive correlation with house value.')

### 2.3 Data Preprocessing & Train-Test Split

In [ ]:
# Separate features (X) and target (y)
X_reg = df_housing.drop('MedHouseVal', axis=1)
y_reg = df_housing['MedHouseVal']

# Split into training (80%) and testing (20%) sets
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

print(f'Training set size: {X_train_reg.shape[0]} samples')
print(f'Testing set size:  {X_test_reg.shape[0]} samples')
print(f'Train/Test ratio:  80/20')

In [ ]:
# Feature Scaling using StandardScaler
scaler_reg = StandardScaler()
X_train_reg_scaled = scaler_reg.fit_transform(X_train_reg)
X_test_reg_scaled = scaler_reg.transform(X_test_reg)

print('✅ Features scaled using StandardScaler (mean=0, std=1)')

### 2.4 Train Linear Regression Model

In [ ]:
# Initialize and train the model
lr_model = LinearRegression()
lr_model.fit(X_train_reg_scaled, y_train_reg)

# Make predictions
y_pred_reg = lr_model.predict(X_test_reg_scaled)

print('✅ Linear Regression model trained successfully!')
print(f'\nModel Intercept: {lr_model.intercept_:.4f}')
print(f'\nModel Coefficients:')
coef_df = pd.DataFrame({
    'Feature': housing.feature_names,
    'Coefficient': lr_model.coef_
}).sort_values('Coefficient', ascending=False)
coef_df

### 2.5 Evaluate Regression Model

In [ ]:
# Calculate evaluation metrics
mse = mean_squared_error(y_test_reg, y_pred_reg)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test_reg, y_pred_reg)
r2 = r2_score(y_test_reg, y_pred_reg)

print('=' * 45)
print('  📈 LINEAR REGRESSION — EVALUATION METRICS')
print('=' * 45)
print(f'  Mean Squared Error (MSE):   {mse:.4f}')
print(f'  Root Mean Squared Error:    {rmse:.4f}')
print(f'  Mean Absolute Error (MAE):  {mae:.4f}')
print(f'  R² Score:                   {r2:.4f}')
print('=' * 45)
print(f'\n💡 The model explains {r2*100:.1f}% of the variance in house prices.')

In [ ]:
# Visualization: Actual vs Predicted values
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot
axes[0].scatter(y_test_reg, y_pred_reg, alpha=0.3, s=10, color='steelblue')
axes[0].plot([0, 5], [0, 5], 'r--', linewidth=2, label='Perfect Prediction')
axes[0].set_xlabel('Actual Values ($100k)')
axes[0].set_ylabel('Predicted Values ($100k)')
axes[0].set_title(f'Actual vs. Predicted (R² = {r2:.3f})')
axes[0].legend()

# Residual distribution
residuals = y_test_reg - y_pred_reg
axes[1].hist(residuals, bins=50, color='coral', edgecolor='white')
axes[1].axvline(0, color='black', linestyle='--', linewidth=1.5)
axes[1].set_xlabel('Residual (Actual - Predicted)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Residual Distribution')

plt.tight_layout()
plt.show()

print('📊 Residuals are approximately normally distributed — a good sign for linear regression!')

---
## 3. Part B — Supervised Learning: Classification
### Classifying Iris Species with Multiple Algorithms

We'll use the classic **Iris dataset** to classify flower species (Setosa, Versicolor, Virginica) based on sepal and petal measurements. We'll compare **Logistic Regression**, **Decision Tree**, and **Random Forest**.

### 3.1 Load & Explore the Iris Dataset

In [ ]:
# Load the Iris dataset
iris = load_iris()
df_iris = pd.DataFrame(iris.data, columns=iris.feature_names)
df_iris['species'] = pd.Categorical.from_codes(iris.target, iris.target_names)

print(f'Dataset Shape: {df_iris.shape}')
print(f'Features: {list(iris.feature_names)}')
print(f'Classes: {list(iris.target_names)}')
print()
df_iris.head(10)

In [ ]:
# Class distribution
print('Class Distribution:')
print(df_iris['species'].value_counts())
print('\n✅ Perfectly balanced dataset — 50 samples per class.')

### 3.2 EDA — Visualizing Feature Distributions

In [ ]:
# Pairplot to visualize feature relationships by species
sns.pairplot(df_iris, hue='species', palette='Set2', diag_kind='hist',
             plot_kws={'alpha': 0.7, 's': 40})
plt.suptitle('Iris Dataset — Feature Pairplot', y=1.02, fontsize=14, fontweight='bold')
plt.show()

print('📊 Key Insight: Setosa is linearly separable. Versicolor and Virginica overlap slightly.')

In [ ]:
# Box plots for each feature
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for i, feature in enumerate(iris.feature_names):
    sns.boxplot(x='species', y=feature, data=df_iris, ax=axes[i], palette='Set2')
    axes[i].set_title(feature.replace(' (cm)', ''), fontsize=11)
    axes[i].set_xlabel('')

plt.suptitle('Feature Distributions by Species', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 3.3 Data Preprocessing & Train-Test Split

In [ ]:
# Separate features and target
X_cls = iris.data
y_cls = iris.target

# Train-Test Split (80/20)
X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(
    X_cls, y_cls, test_size=0.2, random_state=42, stratify=y_cls
)

# Feature Scaling
scaler_cls = StandardScaler()
X_train_cls_scaled = scaler_cls.fit_transform(X_train_cls)
X_test_cls_scaled = scaler_cls.transform(X_test_cls)

print(f'Training set: {X_train_cls_scaled.shape[0]} samples')
print(f'Testing set:  {X_test_cls_scaled.shape[0]} samples')
print('✅ Data split and scaled successfully!')

### 3.4 Train & Compare Classification Models

In [ ]:
# Define models
models = {
    'Logistic Regression': LogisticRegression(max_iter=200, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=4, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42)
}

# Train each model and store results
results = {}

for name, model in models.items():
    model.fit(X_train_cls_scaled, y_train_cls)
    y_pred = model.predict(X_test_cls_scaled)
    accuracy = accuracy_score(y_test_cls, y_pred)
    results[name] = {
        'model': model,
        'predictions': y_pred,
        'accuracy': accuracy
    }
    print(f'✅ {name:25s} — Accuracy: {accuracy*100:.1f}%')

print('\n🏆 All models trained and evaluated!')

### 3.5 Model Evaluation — Detailed Metrics

In [ ]:
# Classification Report for each model
for name, result in results.items():
    print('\n' + '=' * 55)
    print(f'  📋 {name} — Classification Report')
    print('=' * 55)
    print(classification_report(
        y_test_cls, result['predictions'],
        target_names=iris.target_names
    ))

In [ ]:
# Confusion Matrices for all models
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (name, result) in enumerate(results.items()):
    cm = confusion_matrix(y_test_cls, result['predictions'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                xticklabels=iris.target_names, yticklabels=iris.target_names,
                cbar=False, linewidths=1, linecolor='white')
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('Actual')
    axes[idx].set_title(f'{name}\nAccuracy: {result["accuracy"]*100:.1f}%', fontweight='bold')

plt.suptitle('Confusion Matrices — Model Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Model Accuracy Comparison Bar Chart
model_names = list(results.keys())
accuracies = [results[name]['accuracy'] * 100 for name in model_names]
colors = ['#4C72B0', '#55A868', '#C44E52']

plt.figure(figsize=(10, 5))
bars = plt.bar(model_names, accuracies, color=colors, edgecolor='white', linewidth=1.5)

for bar, acc in zip(bars, accuracies):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{acc:.1f}%', ha='center', fontweight='bold', fontsize=13)

plt.ylim(0, 110)
plt.ylabel('Accuracy (%)')
plt.title('Model Accuracy Comparison — Iris Classification', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 3.6 Feature Importance (Random Forest)

In [ ]:
# Extract feature importance from Random Forest
rf_model = results['Random Forest']['model']
feature_importance = pd.DataFrame({
    'Feature': iris.feature_names,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=True)

plt.figure(figsize=(10, 5))
plt.barh(feature_importance['Feature'], feature_importance['Importance'],
         color='#55A868', edgecolor='white', linewidth=1.5)
plt.xlabel('Feature Importance Score')
plt.title('Random Forest — Feature Importance', fontsize=14, fontweight='bold')

for i, v in enumerate(feature_importance['Importance']):
    plt.text(v + 0.005, i, f'{v:.3f}', va='center', fontweight='bold')

plt.tight_layout()
plt.show()

print('📊 Petal length and petal width are the most important features for classification.')

---
## 4. Key Concepts Summary

| Concept | Description |
|---------|-------------|
| **Supervised Learning** | Model learns from labeled data (input → output mapping) |
| **Regression** | Predicts a continuous value (e.g., house price) |
| **Classification** | Predicts a discrete class/category (e.g., flower species) |
| **Train-Test Split** | Dividing data into training (80%) and testing (20%) sets to evaluate generalization |
| **Feature Scaling** | Standardizing features (mean=0, std=1) so all features contribute equally |
| **R² Score** | Measures how much variance the model explains (1.0 = perfect) |
| **RMSE** | Root Mean Squared Error — average prediction error in same units as target |
| **Accuracy** | Percentage of correctly classified samples |
| **Confusion Matrix** | Shows true vs. predicted labels for each class |
| **Feature Importance** | Ranks which features contribute most to predictions |

---
## 5. Conclusion

### Regression Results
- **Linear Regression** achieved a solid R² score on the California Housing dataset.
- **Median Income** was the strongest predictor of house prices.
- Residuals were approximately normal, validating our model assumptions.

### Classification Results
- All three models achieved high accuracy on the Iris dataset.
- **Petal length** and **petal width** were the most discriminative features.
- **Random Forest** typically provides the most robust performance due to ensemble averaging.

### Key Takeaways
- Always split data before scaling to prevent **data leakage**.
- Use **multiple evaluation metrics** — accuracy alone can be misleading.
- **Feature importance** analysis helps understand what drives model predictions.
- Start with simple models (Linear/Logistic Regression) as baselines before trying complex ones.

---
*Notebook by Mohd Anas — Data Science Internship, Codomax Academy (September 2026)*